# 22-26 · SQL i jego wyświetlanie w ORM

Praktyka do sekcji [„ORM: pracy z bazą danych przez Python obiekty”

## Cel

Sprawdź, czy zapytanie przez obiekt zabawkowy ORM- oraz bezpośrednie żądanie SQL- dają ten sam wynik — używając małego, całkowicie realistycznego przykładu sqlite3.

## Sprawa robocza

In [ ]:
import sqlite3

baza = sqlite3.connect(":memory:")
baza.row_factory = sqlite3.Row
baza.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)")
baza.executemany(
    "INSERT INTO tasks (title, done) VALUES (?, ?)",
    [("A", 0), ("B", 1), ("C", 0)],
)
baza.commit()


class ProstoyORM:
    """Игрушечная имитация одного метода ORM: формирует и выполняет обычный SQL."""

    def __init__(self, soedinenie):
        self.soedinenie = soedinenie

    def nevypolnennye(self):
        return self.soedinenie.execute(
            "SELECT id, title, done FROM tasks WHERE done = 0 ORDER BY id"
        ).fetchall()


orm = ProstoyORM(baza)
cherez_orm = orm.nevypolnennye()
cherez_sql = baza.execute("SELECT id, title, done FROM tasks WHERE done = 0 ORDER BY id").fetchall()

print([dict(s) for s in cherez_orm])

## Sprawdzenie wyniku

In [ ]:
assert [dict(s) for s in cherez_orm] == [dict(s) for s in cherez_sql]
assert [s["title"] for s in cherez_orm] == ["A", "C"]
print("Верно: ORM-обёртка и прямой SQL-запрос вернули одинаковый результат — ORM формирует тот же SQL.")